In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import os
from pathlib import Path

In [ ]:


class GermanElectionScraper:
    def __init__(self, save_html=True):
        self.base_url = "https://www.bundeswahlleiterin.de/bundestagswahlen/2025/ergebnisse/bund-99"
        self.save_html = save_html
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        })
        
        # Create directories for saved HTML
        if self.save_html:
            Path("html/states").mkdir(parents=True, exist_ok=True)
            Path("html/constituencies").mkdir(parents=True, exist_ok=True)
        
        # State mapping with constituency ranges
        self.states = {
            1: {"name": "Schleswig-Holstein", "constituencies": range(1, 12)},
            2: {"name": "Hamburg", "constituencies": range(18, 24)},
            3: {"name": "Lower Saxony", "constituencies": range(24, 54)},
            4: {"name": "Bremen", "constituencies": range(54, 56)},
            5: {"name": "North Rhine-Westphalia", "constituencies": range(86, 150)},
            6: {"name": "Hesse", "constituencies": range(166, 188)},
            7: {"name": "Rhineland-Palatinate", "constituencies": range(196, 211)},
            8: {"name": "Baden-Württemberg", "constituencies": range(258, 296)},
            9: {"name": "Bavaria", "constituencies": range(211, 258)},
            10: {"name": "Saarland", "constituencies": range(296, 300)},
            11: {"name": "Berlin", "constituencies": range(74, 86)},
            12: {"name": "Brandenburg", "constituencies": range(56, 66)},
            13: {"name": "Mecklenburg-Western Pomerania", "constituencies": range(12, 18)},
            14: {"name": "Saxony", "constituencies": range(150, 166)},
            15: {"name": "Saxony-Anhalt", "constituencies": range(66, 74)},
            16: {"name": "Thuringia", "constituencies": range(188, 196)}
        }
    
    def save_html_content(self, content, filename, folder):
        """Save HTML content to file"""
        if self.save_html:
            filepath = Path(f"html/{folder}") / filename
            with open(filepath, 'w', encoding='utf-8') as f:
                f.write(content)
    
    def get_state_data(self, state_number):
        """Scrape data for a specific state"""
        url = f"{self.base_url}/land-{state_number}.html"
        
        try:
            response = self.session.get(url)
            response.raise_for_status()
            
            # Save HTML
            self.save_html_content(response.text, f"state_{state_number}.html", "states")
            
            soup = BeautifulSoup(response.content, 'html.parser')
            
            state_data = {
                'state_number': state_number,
                'state_name': self.states[state_number]["name"],
                'url': url
            }
            
            # Extract state-level voting data
            state_data.update(self.extract_voting_data(soup))
            
            return state_data
            
        except Exception as e:
            print(f"Error scraping state {state_number}: {e}")
            return None
    
    def get_constituency_data(self, state_number, constituency_number):
        """Scrape data for a specific constituency"""
        url = f"{self.base_url}/land-{state_number}/wahlkreis-{constituency_number}.html"
        
        try:
            response = self.session.get(url)
            response.raise_for_status()
            
            # Save HTML
            self.save_html_content(response.text, f"state_{state_number}_wk_{constituency_number}.html", "constituencies")
            
            soup = BeautifulSoup(response.content, 'html.parser')
            
            constituency_data = {
                'state_number': state_number,
                'state_name': self.states[state_number]["name"],
                'constituency_number': constituency_number,
                'url': url
            }
            
            # Extract constituency name properly
            constituency_name = self.extract_constituency_name(soup)
            constituency_data['constituency_name'] = constituency_name
            
            # Extract detailed voting data
            constituency_data.update(self.extract_voting_data(soup))
            
            return constituency_data
            
        except Exception as e:
            print(f"Error scraping constituency {constituency_number} in state {state_number}: {e}")
            return None
    
    def extract_constituency_name(self, soup):
        """Extract constituency name from the page"""
        # Try multiple possible selectors for constituency name
        selectors = [
            'h1',
            '.page-title',
            '.wahlkreis-name',
            '[class*="wahlkreis"] h1',
            '[class*="wahlkreis"] h2'
        ]
        
        for selector in selectors:
            element = soup.select_one(selector)
            if element:
                text = element.get_text(strip=True)
                # Filter out generic text and look for actual constituency names
                if text and len(text) > 10 and any(keyword in text.lower() for keyword in ['wahlkreis', 'constituency']):
                    # Clean up the name
                    name = re.sub(r'Wahlkreis\s*\d+\s*[-:]?\s*', '', text).strip()
                    if name and len(name) > 3:
                        return name
        
        # Fallback: look for any h1 that seems like a constituency name
        h1_elements = soup.find_all('h1')
        for h1 in h1_elements:
            text = h1.get_text(strip=True)
            if text and len(text) > 10 and not any(generic in text.lower() for generic in ['bundestagswahl', 'ergebnisse']):
                return text
        
        return "Unknown Constituency"
    
    def extract_voting_data(self, soup):
        """Extract the detailed voting table data like in the screenshot"""
        voting_data = {}
        
        # Find all tables on the page
        tables = soup.find_all('table')
        
        for i, table in enumerate(tables):
            # Look for tables that might contain the voting data
            if self.is_voting_table(table):
                voting_data.update(self.parse_voting_table(table))
                break  # Stop after finding the first voting table
        
        return voting_data
    
    def is_voting_table(self, table):
        """Check if this table contains voting data"""
        # Look for headers that indicate voting data
        headers = table.get_text()
        voting_indicators = ['erststimme', 'zweitstimme', 'erststimmen', 'zweitstimmen', 
                           'first votes', 'second votes', 'wahlberechtigte', 'eligible voters']
        
        return any(indicator in headers.lower() for indicator in voting_indicators)
    
    def parse_voting_table(self, table):
        """Parse the voting table with first and second votes"""
        data = {}
        
        # Extract all rows
        rows = table.find_all('tr')
        
        # Party mapping for consistent naming
        party_mapping = {
            'spd': 'SPD',
            'cdu': 'CDU',
            'grüne': 'GREENS',
            'green': 'GREENS', 
            'fdp': 'FDP',
            'afd': 'AFD',
            'linke': 'THE_LEFT',
            'left': 'THE_LEFT',
            'bsw': 'BSW',
            'fw': 'FREE_VOTERS',
            'freie wähler': 'FREE_VOTERS',
            'volt': 'VOLT',
            'mlpd': 'MLPD',
            'ödp': 'ODP',
            'basis': 'THE_BASIS',
            'partei': 'THE_PARTY'
        }
        
        for row in rows:
            cells = [cell.get_text(strip=True) for cell in row.find_all(['td', 'th'])]
            
            if len(cells) >= 5:
                # This looks like a data row
                row_name = cells[0].lower()
                
                # Parse voter statistics
                if 'wahlberechtigte' in row_name or 'eligible voters' in row_name:
                    data['eligible_voters'] = self.clean_number(cells[1])
                elif 'wähler' in row_name or 'voters' in row_name:
                    data['voters'] = self.clean_number(cells[1])
                    data['voter_turnout_percent'] = self.clean_number(cells[2])
                elif 'ungültige' in row_name or 'invalid' in row_name:
                    data['invalid_votes_first'] = self.clean_number(cells[1])
                    data['invalid_votes_first_percent'] = self.clean_number(cells[2])
                    data['invalid_votes_second'] = self.clean_number(cells[3])
                    data['invalid_votes_second_percent'] = self.clean_number(cells[4])
                elif 'gültige' in row_name or 'valid' in row_name:
                    data['valid_votes_first'] = self.clean_number(cells[1])
                    data['valid_votes_first_percent'] = self.clean_number(cells[2])
                    data['valid_votes_second'] = self.clean_number(cells[3])
                    data['valid_votes_second_percent'] = self.clean_number(cells[4])
                else:
                    # This is likely a political party row
                    party_name = self.map_party_name(cells[0], party_mapping)
                    if party_name:
                        data[f'{party_name}_first_votes'] = self.clean_number(cells[1])
                        data[f'{party_name}_first_percent'] = self.clean_number(cells[2])
                        data[f'{party_name}_second_votes'] = self.clean_number(cells[3])
                        data[f'{party_name}_second_percent'] = self.clean_number(cells[4])
        
        return data
    
    def map_party_name(self, raw_name, party_mapping):
        """Map raw party names to consistent names"""
        raw_lower = raw_name.lower()
        for key, value in party_mapping.items():
            if key in raw_lower:
                return value
        return None
    
    def clean_number(self, text):
        """Clean and convert number strings"""
        if not text or text == '-' or text == '–':
            return None
        
        # Remove common separators and convert to float/int
        cleaned = re.sub(r'[^\d,\-\.]', '', text.replace('.', '').replace(',', '.'))
        
        # Handle percentage points (might have + or - signs)
        if cleaned and any(char in cleaned for char in ['+', '-']):
            try:
                return float(cleaned)
            except ValueError:
                return cleaned
        
        try:
            if '.' in cleaned:
                return float(cleaned)
            else:
                return int(cleaned) if cleaned else None
        except ValueError:
            return text.strip()
    
    def scrape_all_data(self, delay=1):
        """Scrape all states and constituencies"""
        all_state_data = []
        all_constituency_data = []
        
        print("Starting to scrape German election data...")
        print("HTML files will be saved in 'html/' directory")
        
        for state_num, state_info in self.states.items():
            print(f"Scraping state: {state_info['name']} ({state_num}/16)")
            
            # Scrape state data
            state_data = self.get_state_data(state_num)
            if state_data:
                all_state_data.append(state_data)
            
            # Scrape all constituencies for this state
            for constituency_num in state_info['constituencies']:
                print(f"  Scraping constituency: {constituency_num:03d}")
                
                constituency_data = self.get_constituency_data(state_num, constituency_num)
                if constituency_data:
                    all_constituency_data.append(constituency_data)
                
                time.sleep(delay)  # Be polite to the server
        
        return all_state_data, all_constituency_data
    
    def save_to_excel(self, state_data, constituency_data, filename="german_election_data.xlsx"):
        """Save data to Excel file"""
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            # State data
            if state_data:
                state_df = pd.DataFrame(state_data)
                state_df.to_excel(writer, sheet_name='States', index=False)
            
            # Constituency data
            if constituency_data:
                constituency_df = pd.DataFrame(constituency_data)
                constituency_df.to_excel(writer, sheet_name='Constituencies', index=False)
        
        print(f"Data saved to {filename}")

# Usage
def main():
    scraper = GermanElectionScraper(save_html=True)
    
    # Scrape all data
    state_data, constituency_data = scraper.scrape_all_data(delay=0.5)
    
    # Save to Excel
    scraper.save_to_excel(state_data, constituency_data, "german_election_2025_detailed.xlsx")
    
    # Print summary
    print(f"\nScraping completed!")
    print(f"States scraped: {len(state_data)}")
    print(f"Constituencies scraped: {len(constituency_data)}")
    
    # Show sample of what was collected
    if constituency_data:
        sample = constituency_data[0]
        print(f"\nSample constituency data keys: {list(sample.keys())[:10]}...")

if __name__ == "__main__":
    main()